In [28]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import joblib


In [2]:
clinical_text_data = pd.read_csv(r"C:\Users\jagad\Documents\my_classes\Tasks\mini_project_5-Clinical_Trial_Disease_Category_Classification\data\preprocessed_data\cleaned_text_data.csv")

clinical_text_data.head()

,source_condition_query,brief_summary,cleaned_summary
0,breast cancer,Breast cancer patients often have perioperativ...,breast cancer patient often perioperative emot...
1,breast cancer,Many breast cancer patients experience psychol...,many breast cancer patient experience psycholo...
2,breast cancer,"Based on an American study by Scherer et al., ...",base american study scherer et al. hypothesize...
3,breast cancer,Compare the effect of ropivacaine versus place...,compare effect ropivacaine versus placebo pect...
4,breast cancer,Phase 1 dose escalation and expansion study of...,phase 1 dose escalation expansion study clsp-1...


### Encoding:

In [3]:
print("label encoding is done source_condition_query:")
clinical_text_encoded = clinical_text_data.copy()
le = LabelEncoder()
clinical_text_encoded['source_condition_query_encoded'] = le.fit_transform(clinical_text_encoded['source_condition_query'])

print("Data Shape: ", clinical_text_encoded.shape)

print("\nFirst five Records: ")
print(clinical_text_encoded.head())

print("\nInformation: ")
print(clinical_text_encoded.info())

print("\nMissing values:")
print(clinical_text_encoded.isnull().sum())





label encoding is done source_condition_query:
Data Shape:  (60072, 4)

First five Records: 
  source_condition_query                                      brief_summary  \
0          breast cancer  Breast cancer patients often have perioperativ...   
1          breast cancer  Many breast cancer patients experience psychol...   
2          breast cancer  Based on an American study by Scherer et al., ...   
3          breast cancer  Compare the effect of ropivacaine versus place...   
4          breast cancer  Phase 1 dose escalation and expansion study of...   

                                     cleaned_summary  \
0  breast cancer patient often perioperative emot...   
1  many breast cancer patient experience psycholo...   
2  base american study scherer et al. hypothesize...   
3  compare effect ropivacaine versus placebo pect...   
4  phase 1 dose escalation expansion study clsp-1...   

   source_condition_query_encoded  
0                               1  
1                      

### TF-IDF

In [34]:
tfidf_vectorizer = TfidfVectorizer(
                                    max_features=5000,   # having records around 60,000, so taking sample 5000 is enough
                                    ngram_range=(1, 2),  # keeps single words AND two-word phrases.
                                    min_df=5,            # drops words/phrases appearing in fewer than 5 documents.
                                    max_df=0.9           # drops words/phrases appearing in more than 90% of documents.
)

# Fit and transform:
X_tfidf = tfidf_vectorizer.fit_transform(clinical_text_encoded['cleaned_summary'])

# inspect the result
print("Shape of TF-IDF matrix:", X_tfidf.shape)
print("\nSample of vocabulary learned:")
print(list(tfidf_vectorizer.get_feature_names_out())[:20])

joblib.dump(tfidf_vectorizer, r"C:\Users\jagad\Documents\my_classes\Tasks\mini_project_5-Clinical_Trial_Disease_Category_Classification\models\tfidf_vectorizer.pkl")
print("\ntfidf_vectorizer is saved as pkl file")


Shape of TF-IDF matrix: (60072, 5000)

Sample of vocabulary learned:
['000', '001', '01', '05', '10', '10 day', '10 mg', '10 minute', '10 week', '10 year', '100', '100 mg', '1000', '11', '12', '12 month', '12 week', '12 year', '120', '13']

tfidf_vectorizer is saved as pkl file


### sample one and two words
1. Unigrams (1 word)
2. Bigrams (2 consecutive words)

In [5]:
import random
vocab = tfidf_vectorizer.get_feature_names_out()
sample_indices = random.sample(range(len(vocab)), 20)
print([vocab[i] for i in sample_indices])

['relatively', 'training', 'find', 'contribution', 'intervention improve', 'research show', 'hence', 'steroid', 'bacterial', 'similar', 'inventory', 'apart', 'primary open', 'assume', 'collaboration', 'instruction', 'intranasal', 'inhaler', 'unit icu', 'tomosynthesis']


#### Sample words and its numbers

In [6]:
doc_index = 0
row = X_tfidf[doc_index].toarray().flatten()
vocab = tfidf_vectorizer.get_feature_names_out()

nonzero_indices = row.nonzero()[0]
for idx in nonzero_indices[:15]:
    print(f"{vocab[idx]}: {row[idx]:.4f}")

aim: 0.0556
aim determine: 0.1252
also: 0.0721
anxiety: 0.1437
anxiety depression: 0.2256
breast: 0.1171
breast cancer: 0.1225
bring: 0.1535
cancer: 0.1135
cancer patient: 0.1888
concern: 0.1183
could: 0.1903
could improve: 0.3229
depression: 0.1859
determine: 0.0687


### Train test split

In [7]:
# Features (X) and target (y)
X = X_tfidf
y = clinical_text_encoded['source_condition_query_encoded']

# Stratified 80/20 split
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2,  random_state=42, 
                                    stratify=y    # preserves class proportions in both train and test sets
)

# Verify the split
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)


X_train shape: (48057, 5000)
X_test shape: (12015, 5000)
y_train shape: (48057,)
y_test shape: (12015,)


### Model Training:

#### 1. Naive Bayes model:

In [30]:
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)
nb_test_predictions = nb_model.predict(X_test)
nb_train_predictions = nb_model.predict(X_train)

print("Naive Bayes training complete")

joblib.dump(nb_model,r"C:\Users\jagad\Documents\my_classes\Tasks\mini_project_5-Clinical_Trial_Disease_Category_Classification\models\nb_model.pkl" )
print("nb model saved as pkl file")


Naive Bayes training complete
nb model saved as pkl file


#### 2. Logestic Regression:


In [31]:
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)
lr_test_predictions = lr_model.predict(X_test)
lr_train_predictions = lr_model.predict(X_train)

print("Logistic Regression training complete")

joblib.dump(lr_model,r"C:\Users\jagad\Documents\my_classes\Tasks\mini_project_5-Clinical_Trial_Disease_Category_Classification\models\lr_model.pkl" )
print("lr model saved as pkl file")


Logistic Regression training complete
lr model saved as pkl file


#### 3. Linear SVM

In [32]:
svm_model = LinearSVC(random_state=42)
svm_model.fit(X_train, y_train)
svm_test_predictions = svm_model.predict(X_test)
svm_train_predictions = svm_model.predict(X_train)

print("Linear SVM training complete")

joblib.dump(svm_model,r"C:\Users\jagad\Documents\my_classes\Tasks\mini_project_5-Clinical_Trial_Disease_Category_Classification\models\svm_model.pkl" )
print("svm model saved as pkl file")

Linear SVM training complete
svm model saved as pkl file


### Model Evaluation

#### 1. Naive Bayes Results

In [ ]:
# Accuracy:
train_accuracy_nb = accuracy_score(y_train, nb_train_predictions)
test_accuracy_nb = accuracy_score(y_test, nb_test_predictions)

# Precision:
train_precision_nb = precision_score(y_train, nb_train_predictions, average='macro')
test_precision_nb = precision_score(y_test, nb_test_predictions, average='macro')

# Recall:
train_recall_nb = recall_score(y_train, nb_train_predictions, average='macro')
test_recall_nb = recall_score(y_test, nb_test_predictions, average='macro')

# F1 Score:
train_f1_nb = f1_score(y_train, nb_train_predictions, average='macro')
test_f1_nb = f1_score(y_test, nb_test_predictions, average='macro')


nb_metrics_df = pd.DataFrame({
                        'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
                        'Training Set': [train_accuracy_nb, train_precision_nb, train_recall_nb, train_f1_nb],
                        'Testing Set': [test_accuracy_nb, test_precision_nb, test_recall_nb, test_f1_nb]
})


nb_metrics_df['Training Set'] = nb_metrics_df['Training Set'].apply(lambda x: f"{x:.4f} ({x*100:.2f}%)")
nb_metrics_df['Testing Set'] = nb_metrics_df['Testing Set'].apply(lambda x: f"{x:.4f} ({x*100:.2f}%)")

print("="*50)
print("Naive Bayes Evaluation Metrics:")
print("="*50)
display(nb_metrics_df)



Naive Bayes Evaluation Metrics:


,Metric,Training Set,Testing Set
0,Accuracy,0.9129 (91.29%),0.9074 (90.74%)
1,Precision,0.9266 (92.66%),0.9243 (92.43%)
2,Recall,0.8890 (88.90%),0.8764 (87.64%)
3,F1-Score,0.9051 (90.51%),0.8965 (89.65%)


#### 2. Logestic regression results

In [ ]:
# Accuracy:
train_accuracy_lr = accuracy_score(y_train, lr_train_predictions)
test_accuracy_lr = accuracy_score(y_test, lr_test_predictions)

# Precision:
train_precision_lr = precision_score(y_train, lr_train_predictions, average='macro')
test_precision_lr = precision_score(y_test, lr_test_predictions, average='macro')

# Recall:
train_recall_lr = recall_score(y_train, lr_train_predictions, average='macro')
test_recall_lr = recall_score(y_test, lr_test_predictions, average='macro')

# F1 Score:
train_f1_lr = f1_score(y_train, lr_train_predictions, average='macro')
test_f1_lr = f1_score(y_test, lr_test_predictions, average='macro')


lr_metrics_df = pd.DataFrame({
                        'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
                        'Training Set': [train_accuracy_lr, train_precision_lr, train_recall_lr, train_f1_lr],
                        'Testing Set': [test_accuracy_lr, test_precision_lr, test_recall_lr, test_f1_lr]
})


lr_metrics_df['Training Set'] = lr_metrics_df['Training Set'].apply(lambda x: f"{x:.4f} ({x*100:.2f}%)")
lr_metrics_df['Testing Set'] = lr_metrics_df['Testing Set'].apply(lambda x: f"{x:.4f} ({x*100:.2f}%)")

print("="*50)
print("Logestic regression Evaluation Metrics:")
print("="*50)
display(lr_metrics_df)



Logestic regression Evaluation Metrics:


,Metric,Training Set,Testing Set
0,Accuracy,0.9643 (96.43%),0.9434 (94.34%)
1,Precision,0.9702 (97.02%),0.9501 (95.01%)
2,Recall,0.9480 (94.80%),0.9209 (92.09%)
3,F1-Score,0.9585 (95.85%),0.9344 (93.44%)


### Linear SVM Results:

In [13]:
# Accuracy:
train_accuracy_svm = accuracy_score(y_train, svm_train_predictions)
test_accuracy_svm = accuracy_score(y_test, svm_test_predictions)

# Precision:
train_precision_svm = precision_score(y_train, svm_train_predictions, average='macro')
test_precision_svm = precision_score(y_test, svm_test_predictions, average='macro')

# Recall:
train_recall_svm = recall_score(y_train, svm_train_predictions, average='macro')
test_recall_svm = recall_score(y_test, svm_test_predictions, average='macro')

# F1 Score:
train_f1_svm = f1_score(y_train, svm_train_predictions, average='macro')
test_f1_svm = f1_score(y_test, svm_test_predictions, average='macro')


svm_metrics_df = pd.DataFrame({
                        'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
                        'Training Set': [train_accuracy_svm, train_precision_svm, train_recall_svm, train_f1_svm],
                        'Testing Set': [test_accuracy_svm, test_precision_svm, test_recall_svm, test_f1_svm]
})


svm_metrics_df['Training Set'] = svm_metrics_df['Training Set'].apply(lambda x: f"{x:.4f} ({x*100:.2f}%)")
svm_metrics_df['Testing Set'] = svm_metrics_df['Testing Set'].apply(lambda x: f"{x:.4f} ({x*100:.2f}%)")

print("="*50)
print("Linear SVM Evaluation Metrics:")
print("="*50)
display(svm_metrics_df)



Linear SVM Evaluation Metrics:


,Metric,Training Set,Testing Set
0,Accuracy,0.9875 (98.75%),0.9459 (94.59%)
1,Precision,0.9891 (98.91%),0.9497 (94.97%)
2,Recall,0.9859 (98.59%),0.9287 (92.87%)
3,F1-Score,0.9875 (98.75%),0.9385 (93.85%)


### Overall result model comparison:

In [14]:
# Naive Bayes Evaluation Metrics:
print("="*50)
print("Naive Bayes Evaluation Metrics:")
print("="*50)
display(nb_metrics_df)

# Logestic regression Evaluation Metrics:
print("="*50)
print("Logestic regression Evaluation Metrics:")
print("="*50)
display(lr_metrics_df)

# Linear SVM Evaluation Metrics:
print("="*50)
print("Linear SVM Evaluation Metrics:")
print("="*50)
display(svm_metrics_df)

Naive Bayes Evaluation Metrics:


,Metric,Training Set,Testing Set
0,Accuracy,0.9129 (91.29%),0.9074 (90.74%)
1,Precision,0.9266 (92.66%),0.9243 (92.43%)
2,Recall,0.8890 (88.90%),0.8764 (87.64%)
3,F1-Score,0.9051 (90.51%),0.8965 (89.65%)


Logestic regression Evaluation Metrics:


,Metric,Training Set,Testing Set
0,Accuracy,0.9643 (96.43%),0.9434 (94.34%)
1,Precision,0.9702 (97.02%),0.9501 (95.01%)
2,Recall,0.9480 (94.80%),0.9209 (92.09%)
3,F1-Score,0.9585 (95.85%),0.9344 (93.44%)


Linear SVM Evaluation Metrics:


,Metric,Training Set,Testing Set
0,Accuracy,0.9875 (98.75%),0.9459 (94.59%)
1,Precision,0.9891 (98.91%),0.9497 (94.97%)
2,Recall,0.9859 (98.59%),0.9287 (92.87%)
3,F1-Score,0.9875 (98.75%),0.9385 (93.85%)


### Classification reports:

In [15]:

class_names = le.classes_

print("\nClassification Report (per-class) Naive Bays Model:")
print(classification_report(y_test, nb_test_predictions, target_names=class_names))

print("\nClassification Report (per-class) Logestic Regression Model:")
print(classification_report(y_test, lr_test_predictions, target_names=class_names))

print("\nClassification Report (per-class) Linear SVM Model:")
print(classification_report(y_test, svm_test_predictions, target_names=class_names))


Classification Report (per-class) Naive Bays Model:
                                       precision    recall  f1-score   support

                              anxiety       0.78      0.95      0.86      1852
                        breast cancer       0.95      0.93      0.94      3253
chronic obstructive pulmonary disease       0.95      0.81      0.87      1228
                             covid-19       0.93      0.90      0.92      2021
                             glaucoma       0.96      0.86      0.91       430
                 rheumatoid arthritis       0.95      0.81      0.88       724
                   sickle cell anemia       0.96      0.80      0.87       227
                      type 2 diabetes       0.91      0.95      0.93      2280

                             accuracy                           0.91     12015
                            macro avg       0.92      0.88      0.90     12015
                         weighted avg       0.91      0.91      0.91     120

### Confusion Matrix

In [27]:
# Confusion matrix plot - Naive Bays Model:
cm_nb = confusion_matrix(y_test, nb_test_predictions)

fig_cm_nb = px.imshow(
                cm_nb,
                text_auto=True,
                color_continuous_scale="reds",
                x=class_names,
                y=class_names,
                labels=dict(x="Predicted", y="Actual", color="Count"),
                title="Confusion Matrix - Naive Bays Model"
)

fig_cm_nb.update_layout(
                width=1000,
                height=800,
                xaxis_tickangle=90,
                template="plotly_white"
)

fig_cm_nb.show()


# Confusion matrix plot - Logestic Regression Model:
cm_lr = confusion_matrix(y_test, lr_test_predictions)

fig_cm_lr = px.imshow(
                cm_lr,
                text_auto=True,
                color_continuous_scale="algae",
                x=class_names,
                y=class_names,
                labels=dict(x="Predicted", y="Actual", color="Count"),
                title="Confusion Matrix - Logestic regression Model"
)

fig_cm_lr.update_layout(
                width=1000,
                height=800,
                xaxis_tickangle=90,
                template="plotly_white"
)

fig_cm_lr.show()



# Confusion matrix plot - Linear SVM Model:
cm_svm = confusion_matrix(y_test, svm_test_predictions)

fig_cm_svm = px.imshow(
                cm_svm,
                text_auto=True,
                color_continuous_scale="sunset",
                x=class_names,
                y=class_names,
                labels=dict(x="Predicted", y="Actual", color="Count"),
                title="Confusion Matrix - Linear SVM Model"
)

fig_cm_svm.update_layout(
                width=1000,
                height=800,
                xaxis_tickangle=90,
                template="plotly_white"
)

fig_cm_svm.show()

